# 두피 상태 분류 — EfficientNet-B0 Original vs Augmented

이 Notebook은 **USB 현미경 기반 두피 상태 분류 모델**을 학습하기 위한 최종 학습용 Notebook입니다.

이번 실험의 핵심은 단순히 하나의 모델을 학습하는 것이 아니라,

```text
Original 데이터셋
        ↓
EfficientNet-B0
        ↓
Original Model

Augmented 데이터셋
        ↓
EfficientNet-B0
        ↓
Augmented Model

        ↓
동일한 원본 Test 데이터로 평가
        ↓
Original vs Augmented 비교
```

를 수행하여 **데이터 증강이 두피 현미경 영상 분류 성능에 실제로 도움이 되는지 확인하는 것**입니다.

---

## 이번 실험의 기본 원칙

### 1. Original

원본 이미지만 사용합니다.

```text
original/
├── train/
├── val/
└── test/
```

### 2. Augmented

Train에는 원본과 증강 이미지가 함께 존재합니다.

```text
augmented/
├── train/    ← 원본 + 증강
├── val/      ← 원본 그대로
└── test/     ← 원본 그대로
```

따라서 Validation과 Test에는 증강 이미지를 넣지 않습니다.

이렇게 해야 Original과 Augmented 모델을 **동일한 성격의 원본 평가 데이터**로 비교할 수 있습니다.

---

## 학습 설정

- Model: EfficientNet-B0
- Pretrained weights: ImageNet
- Input size: 224 × 224
- Batch size: 32
- Epoch: 15
- Optimizer: Adam
- Learning rate: 1e-4
- Early Stopping: 사용하지 않음
- Best model 기준: Validation Accuracy
- 평가 지표: Accuracy, Precision, Recall, F1-score, Macro F1
- 추가 분석: Confusion Matrix

> 참고: 이전 웹캠 피부 모델 학습 Notebook과 동일한 실험 원칙을 유지합니다. 해당 Notebook에서도 15 Epoch를 고정하고 Early Stopping 대신 Validation Accuracy가 가장 높은 Epoch의 모델을 사용하도록 구성되어 있습니다. fileciteturn15file2L1-L1


# 1. 연구 목적

이번 모델은 **웹캠 피부 모델과 별도의 USB 현미경 기반 두피 전문 모델**입니다.

두피는 일반적인 얼굴 전체 이미지보다 모낭, 각질, 피지, 홍반, 농포, 모발 밀도 등의 세부적인 특징을 확대해서 보는 것이 중요하기 때문에 별도의 전문 분류 모델로 구성합니다.

프로젝트 전체에서도 피부와 두피에 대해 질환군별 전문 모델을 구축하고 이를 나중에 라우팅 구조로 연결하는 방향을 사용하고 있습니다. fileciteturn17file15L1-L1

이번 Notebook에서는 이미 전처리가 끝난 `hair_processed.zip`을 이용합니다.

```text
Google Drive
└── MyDrive
    └── hair_dataset
        └── hair_processed.zip
```

ZIP을 Colab으로 가져와 압축을 해제한 뒤 학습합니다.


In [1]:
# ============================================================
# 2. Google Drive 연결
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

print("Google Drive 연결 완료")


Mounted at /content/drive
Google Drive 연결 완료


# 3. 라이브러리 설치 및 import

학습에는 TensorFlow/Keras를 사용하고, 평가에는 scikit-learn의 Classification Report와 Confusion Matrix를 사용합니다.

또한 seaborn을 이용하여 Confusion Matrix를 시각화합니다.

Colab에서 GPU를 사용하면 EfficientNet-B0 학습 시간을 줄일 수 있습니다.

GPU가 필요하다면:

**런타임 → 런타임 유형 변경 → GPU**

로 설정합니다.


In [2]:
# ============================================================
# 3. 라이브러리 준비
# ============================================================

!pip -q install seaborn scikit-learn

import os
import json
import shutil
import zipfile
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

print("TensorFlow:", tf.__version__)
print("사용 가능한 GPU:")
print(tf.config.list_physical_devices("GPU"))


TensorFlow: 2.20.0
사용 가능한 GPU:
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# 4. Google Drive의 ZIP 파일 위치

이번 데이터셋은 다음 위치에 있습니다.

```text
My Drive/
└── hair_dataset/
    └── hair_processed.zip
```

Colab에서는 `/content/drive/MyDrive/`를 통해 Google Drive에 접근합니다.

따라서 실제 경로는:

```text
/content/drive/MyDrive/hair_dataset/hair_processed.zip
```

입니다.

학습 결과는 원본 ZIP과 섞이지 않도록 별도의 결과 폴더에 저장합니다.


In [3]:
# ============================================================
# 4. 경로 및 학습 설정
# ============================================================

DRIVE_DATA_DIR = Path(
    "/content/drive/MyDrive/hair_dataset"
)

DRIVE_ZIP_PATH = (
    DRIVE_DATA_DIR / "hair_processed.zip"
)

# ZIP은 Colab 로컬 영역으로 복사하여 사용
LOCAL_ZIP_PATH = Path(
    "/content/hair_processed.zip"
)

# 압축 해제 위치
EXTRACT_DIR = Path(
    "/content/hair_dataset"
)

# 학습 결과
RESULT_DIR = Path(
    "/content/hair_model_results"
)

# Google Drive 백업 위치
DRIVE_RESULT_DIR = (
    DRIVE_DATA_DIR / "hair_model_results"
)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 1e-4
SEED = 42

AUTOTUNE = tf.data.AUTOTUNE

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("ZIP:", DRIVE_ZIP_PATH)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Seed:", SEED)
print("Early Stopping: 사용하지 않음")


ZIP: /content/drive/MyDrive/hair_dataset/hair_processed.zip
Image size: (224, 224)
Batch size: 32
Epochs: 15
Learning rate: 0.0001
Seed: 42
Early Stopping: 사용하지 않음


# 5. ZIP 파일 확인 및 Colab으로 복사

Google Drive의 ZIP을 바로 학습에 사용하는 것이 아니라 Colab의 `/content` 영역으로 복사합니다.

이렇게 하면 Google Drive의 파일 I/O로 인한 병목을 줄이고, 압축 해제 및 학습 과정은 Colab의 로컬 저장공간에서 수행할 수 있습니다.

압축 파일 자체는 원본 데이터 보관용으로 Google Drive에 그대로 남아 있습니다.


In [4]:
# ============================================================
# 5. ZIP 확인 및 로컬 복사
# ============================================================

if not DRIVE_ZIP_PATH.exists():
    raise FileNotFoundError(
        f"ZIP 파일을 찾을 수 없습니다.\n"
        f"확인 경로: {DRIVE_ZIP_PATH}"
    )

shutil.copy2(
    DRIVE_ZIP_PATH,
    LOCAL_ZIP_PATH
)

print("ZIP 복사 완료")
print("Local ZIP:", LOCAL_ZIP_PATH)
print(
    "Size:",
    f"{LOCAL_ZIP_PATH.stat().st_size / (1024**2):.2f} MB"
)


ZIP 복사 완료
Local ZIP: /content/hair_processed.zip
Size: 3291.55 MB


# 6. ZIP 압축 해제

압축을 푼 뒤 실제 학습에는 ZIP 파일이 아니라 압축 해제된 이미지가 사용됩니다.

ZIP 내부 구조가 다음 두 형태 중 어느 것이더라도 대응할 수 있도록 구성합니다.

### 형태 A

```text
hair_dataset/
└── hair_processed/
    ├── original/
    └── augmented/
```

### 형태 B

```text
hair_dataset/
├── original/
└── augmented/
```

압축 내부에 `hair_processed`가 한 번 더 들어가더라도 다음 단계에서 자동으로 `original`과 `augmented`를 찾아갑니다.

이 방식은 이전 웹캠 학습 Notebook에서 사용한 ZIP 자동 탐색 방식과 동일한 목적입니다. fileciteturn14file13L1-L1


In [5]:
# ============================================================
# 6. ZIP 압축 해제
# ============================================================

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("압축 해제 시작...")

with zipfile.ZipFile(
    LOCAL_ZIP_PATH,
    "r"
) as z:

    members = z.namelist()
    total = len(members)

    for i, member in enumerate(
        members,
        start=1
    ):
        z.extract(
            member,
            EXTRACT_DIR
        )

        if (
            i == 1
            or i % max(1, total // 20) == 0
            or i == total
        ):
            print(
                f"진행: {i:,}/{total:,} "
                f"({i / total * 100:.1f}%)"
            )

print("압축 해제 완료")


압축 해제 시작...
진행: 1/34,839 (0.0%)
진행: 1,741/34,839 (5.0%)
진행: 3,482/34,839 (10.0%)
진행: 5,223/34,839 (15.0%)
진행: 6,964/34,839 (20.0%)
진행: 8,705/34,839 (25.0%)
진행: 10,446/34,839 (30.0%)
진행: 12,187/34,839 (35.0%)
진행: 13,928/34,839 (40.0%)
진행: 15,669/34,839 (45.0%)
진행: 17,410/34,839 (50.0%)
진행: 19,151/34,839 (55.0%)
진행: 20,892/34,839 (60.0%)
진행: 22,633/34,839 (65.0%)
진행: 24,374/34,839 (70.0%)
진행: 26,115/34,839 (75.0%)
진행: 27,856/34,839 (80.0%)
진행: 29,597/34,839 (85.0%)
진행: 31,338/34,839 (90.0%)
진행: 33,079/34,839 (94.9%)
진행: 34,820/34,839 (99.9%)
진행: 34,839/34,839 (100.0%)
압축 해제 완료


# 7. 압축 해제된 데이터 구조 확인

압축 해제가 정상적으로 되었는지 확인합니다.

최종적으로 필요한 구조는:

```text
original/
├── train/
│   ├── 클래스1/
│   ├── 클래스2/
│   └── ...
├── val/
└── test/

augmented/
├── train/
│   ├── 클래스1/
│   ├── 클래스2/
│   └── ...
├── val/
└── test/
```

입니다.

특히 이번 실험에서는 `original`과 `augmented`의 클래스 폴더가 동일해야 합니다.

클래스가 다르거나 `train/val/test` 중 하나가 없으면 학습을 시작하지 않고 오류를 발생시키도록 합니다.


In [6]:
# ============================================================
# 7. original / augmented 자동 탐색
# ============================================================

def find_dataset_root(base_dir, target_name):
    candidates = []

    for root, dirs, files in os.walk(base_dir):
        for d in dirs:
            if d.lower() == target_name.lower():

                candidate = Path(root) / d

                required = [
                    candidate / "train",
                    candidate / "val",
                    candidate / "test"
                ]

                if all(
                    p.is_dir()
                    for p in required
                ):
                    candidates.append(candidate)

    if not candidates:
        raise FileNotFoundError(
            f"{target_name} 폴더의 train/val/test 구조를 "
            f"찾지 못했습니다: {base_dir}"
        )

    return candidates[0]


ORIGINAL_DIR = find_dataset_root(
    EXTRACT_DIR,
    "original"
)

AUGMENTED_DIR = find_dataset_root(
    EXTRACT_DIR,
    "augmented"
)

print("Original:")
print(ORIGINAL_DIR)

print("\nAugmented:")
print(AUGMENTED_DIR)


Original:
/content/hair_dataset/hair_processed/original

Augmented:
/content/hair_dataset/hair_processed/augmented


# 8. 클래스 확인

이번 두피 데이터셋은 전처리 과정에서 **질환명 기준으로 클래스를 통합**한 데이터셋입니다.

현재 프로젝트에서 사용하기로 한 두피 클래스는 다음 6개입니다.

```text
모낭사이홍반
모낭홍반농포
미세각질
비듬
탈모
피지과다
```

다만 Notebook에서는 클래스명을 직접 하드코딩하지 않고 `original/train`의 폴더명을 읽어 자동으로 가져옵니다.

이렇게 하면 폴더의 실제 클래스명과 TensorFlow의 class mapping이 항상 일치합니다.

그리고 `augmented/train`에서도 같은 클래스가 존재하는지 확인합니다.


In [7]:
# ============================================================
# 8. 클래스 및 데이터 개수 확인
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


def get_classes(dataset_dir):
    train_dir = dataset_dir / "train"

    classes = sorted([
        p.name
        for p in train_dir.iterdir()
        if p.is_dir()
    ])

    return classes


def count_images(directory):
    directory = Path(directory)

    if not directory.exists():
        return 0

    return sum(
        1
        for p in directory.rglob("*")
        if (
            p.is_file()
            and p.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    )


CLASS_NAMES = get_classes(
    ORIGINAL_DIR
)

AUGMENTED_CLASS_NAMES = get_classes(
    AUGMENTED_DIR
)

if CLASS_NAMES != AUGMENTED_CLASS_NAMES:
    raise ValueError(
        "Original과 Augmented의 클래스 구성이 다릅니다.\n"
        f"Original : {CLASS_NAMES}\n"
        f"Augmented: {AUGMENTED_CLASS_NAMES}"
    )

NUM_CLASSES = len(CLASS_NAMES)

print("클래스 수:", NUM_CLASSES)
print("클래스 순서:")

for i, name in enumerate(CLASS_NAMES):
    print(f"  {i}: {name}")


print("\n" + "=" * 70)

for dataset_name, dataset_dir in [
    ("Original", ORIGINAL_DIR),
    ("Augmented", AUGMENTED_DIR)
]:

    print(f"\n[{dataset_name}]")

    for split in [
        "train",
        "val",
        "test"
    ]:

        split_dir = dataset_dir / split

        total = count_images(
            split_dir
        )

        print(
            f"  {split:5s}: "
            f"{total:,}장"
        )

        for class_name in CLASS_NAMES:

            n = count_images(
                split_dir / class_name
            )

            print(
                f"    {class_name}: {n:,}"
            )


클래스 수: 5
클래스 순서:
  0: 모낭사이홍반
  1: 미세각질
  2: 비듬
  3: 탈모
  4: 피지과다


[Original]
  train: 11,600장
    모낭사이홍반: 2,320
    미세각질: 2,320
    비듬: 2,320
    탈모: 2,320
    피지과다: 2,320
  val  : 1,450장
    모낭사이홍반: 290
    미세각질: 290
    비듬: 290
    탈모: 290
    피지과다: 290
  test : 1,450장
    모낭사이홍반: 290
    미세각질: 290
    비듬: 290
    탈모: 290
    피지과다: 290

[Augmented]
  train: 17,400장
    모낭사이홍반: 3,480
    미세각질: 3,480
    비듬: 3,480
    탈모: 3,480
    피지과다: 3,480
  val  : 1,450장
    모낭사이홍반: 290
    미세각질: 290
    비듬: 290
    탈모: 290
    피지과다: 290
  test : 1,450장
    모낭사이홍반: 290
    미세각질: 290
    비듬: 290
    탈모: 290
    피지과다: 290


# 9. 데이터 구성 확인

이번 실험에서 가장 중요한 부분입니다.

### Original

```text
Train = 원본
Val   = 원본
Test  = 원본
```

### Augmented

```text
Train = 원본 + 증강
Val   = 원본
Test  = 원본
```

즉 Augmented 모델이라고 해서 Validation/Test까지 증강 데이터를 사용하는 것이 아닙니다.

**증강의 효과는 Train에만 추가된 데이터로 학습했을 때 평가 데이터에 대한 일반화 성능이 어떻게 변하는지를 보는 것**입니다.

따라서 최종 비교는:

```text
Original Model
        ↓
원본 Test

vs

Augmented Model
        ↓
원본 Test
```

가 됩니다.

이 구조가 지켜지지 않으면 두 모델의 Test Accuracy를 공정하게 비교하기 어렵습니다.


In [8]:
# ============================================================
# 9. 데이터셋 구성 검증
# ============================================================

def verify_dataset_structure(dataset_name, dataset_dir):

    print("=" * 70)
    print(dataset_name)
    print("=" * 70)

    for split in ["train", "val", "test"]:

        split_dir = dataset_dir / split

        if not split_dir.exists():
            raise FileNotFoundError(
                f"{dataset_name}/{split} 폴더가 없습니다."
            )

        for class_name in CLASS_NAMES:

            class_dir = (
                split_dir / class_name
            )

            if not class_dir.exists():
                raise FileNotFoundError(
                    f"{dataset_name}/{split}/{class_name} "
                    f"폴더가 없습니다."
                )

            n = count_images(
                class_dir
            )

            if n == 0:
                raise ValueError(
                    f"{dataset_name}/{split}/{class_name} "
                    f"이미지가 0장입니다."
                )

            print(
                f"✓ {split:5s} / "
                f"{class_name:15s} : "
                f"{n:,}장"
            )

    print()


verify_dataset_structure(
    "Original",
    ORIGINAL_DIR
)

verify_dataset_structure(
    "Augmented",
    AUGMENTED_DIR
)

print("데이터 구조 검증 완료")


Original
✓ train / 모낭사이홍반          : 2,320장
✓ train / 미세각질            : 2,320장
✓ train / 비듬              : 2,320장
✓ train / 탈모              : 2,320장
✓ train / 피지과다            : 2,320장
✓ val   / 모낭사이홍반          : 290장
✓ val   / 미세각질            : 290장
✓ val   / 비듬              : 290장
✓ val   / 탈모              : 290장
✓ val   / 피지과다            : 290장
✓ test  / 모낭사이홍반          : 290장
✓ test  / 미세각질            : 290장
✓ test  / 비듬              : 290장
✓ test  / 탈모              : 290장
✓ test  / 피지과다            : 290장

Augmented
✓ train / 모낭사이홍반          : 3,480장
✓ train / 미세각질            : 3,480장
✓ train / 비듬              : 3,480장
✓ train / 탈모              : 3,480장
✓ train / 피지과다            : 3,480장
✓ val   / 모낭사이홍반          : 290장
✓ val   / 미세각질            : 290장
✓ val   / 비듬              : 290장
✓ val   / 탈모              : 290장
✓ val   / 피지과다            : 290장
✓ test  / 모낭사이홍반          : 290장
✓ test  / 미세각질            : 290장
✓ test  / 비듬              : 290장
✓ test  / 탈모              : 290장
✓ t

# 10. TensorFlow Dataset 생성

폴더 구조를 그대로 이용하여 TensorFlow Dataset을 만듭니다.

폴더 이름이 곧 정답 클래스가 됩니다.

예:

```text
train/
├── 모낭사이홍반/
├── 모낭홍반농포/
├── 미세각질/
├── 비듬/
├── 탈모/
└── 피지과다/
```

TensorFlow는 이 폴더명을 이용하여 정수형 class label을 자동으로 생성합니다.

### Shuffle

- Train → `shuffle=True`
- Validation → `shuffle=False`
- Test → `shuffle=False`

Train은 데이터 순서를 섞어서 학습하고, Validation/Test는 평가 재현성을 위해 순서를 섞지 않습니다.

### Prefetch

`prefetch(AUTOTUNE)`을 사용하여 CPU가 다음 데이터를 준비하는 동안 GPU가 현재 batch를 학습할 수 있도록 데이터 파이프라인을 구성합니다.


In [9]:
# ============================================================
# 10. TensorFlow Dataset 생성
# ============================================================

def make_datasets(dataset_dir):

    common_kwargs = dict(
        labels="inferred",
        label_mode="int",
        class_names=CLASS_NAMES,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE
    )

    train_ds = tf.keras.utils.image_dataset_from_directory(
        dataset_dir / "train",
        shuffle=True,
        seed=SEED,
        **common_kwargs
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        dataset_dir / "val",
        shuffle=False,
        **common_kwargs
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        dataset_dir / "test",
        shuffle=False,
        **common_kwargs
    )

    train_ds = train_ds.prefetch(
        AUTOTUNE
    )

    val_ds = val_ds.prefetch(
        AUTOTUNE
    )

    test_ds = test_ds.prefetch(
        AUTOTUNE
    )

    return (
        train_ds,
        val_ds,
        test_ds
    )


# 11. EfficientNet-B0 모델

이번 모델은 **ImageNet pretrained EfficientNet-B0**를 사용합니다.

EfficientNet-B0는 이미지 분류에 사용되는 CNN 계열의 모델로, ImageNet에서 학습된 feature extractor를 가져와 새로운 질환 분류 문제에 적용할 수 있습니다.

이번 모델 구조는:

```text
224 × 224 × 3
       ↓
EfficientNet-B0
       ↓
Global Average Pooling
       ↓
Dropout
       ↓
Dense(두피 클래스 수)
       ↓
Softmax
```

입니다.

### Backbone 동결

초기 학습에서는 ImageNet pretrained backbone을 동결하고 classification head를 학습합니다.

즉, 처음부터 수백만 개의 backbone parameter를 모두 변경하는 것이 아니라, 이미 학습된 시각적 feature를 활용하여 두피 클래스에 맞는 최종 분류기를 학습합니다.

이 구조는 이전 웹캠 피부 모델과 동일한 baseline입니다. fileciteturn16file13L1-L1


In [10]:
# ============================================================
# 11. EfficientNet-B0 모델 생성
# ============================================================

def build_model():

    base_model = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(
            IMAGE_SIZE[0],
            IMAGE_SIZE[1],
            3
        )
    )

    # ImageNet pretrained backbone 동결
    base_model.trainable = False

    inputs = keras.Input(
        shape=(
            IMAGE_SIZE[0],
            IMAGE_SIZE[1],
            3
        )
    )

    x = base_model(
        inputs,
        training=False
    )

    x = layers.GlobalAveragePooling2D()(
        x
    )

    x = layers.Dropout(
        0.3
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )(x)

    model = keras.Model(
        inputs,
        outputs
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


model = build_model()

model.summary()


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │         6,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,055,976 (15.47 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

# 12. 학습 설정 — Early Stopping을 사용하지 않는 이유

이번 실험에서는 **Original과 Augmented를 동일한 조건에서 비교하기 위해 15 Epoch를 정확히 수행**합니다.

Early Stopping을 사용하면 데이터셋마다 Validation Accuracy가 상승하는 속도가 다르기 때문에 한 모델은 7 Epoch에서 종료되고 다른 모델은 12 Epoch에서 종료되는 식의 차이가 생길 수 있습니다.

이번 실험에서는 이를 피하기 위해:

```text
Original   → 15 Epoch
Augmented  → 15 Epoch
```

로 고정합니다.

다만 15번째 Epoch의 모델을 무조건 최종 모델로 사용하는 것은 아닙니다.

각 Epoch의 Validation Accuracy를 확인하고:

```text
Epoch 1  → val_accuracy
Epoch 2  → val_accuracy
...
Epoch 15 → val_accuracy
             ↓
     가장 높은 Epoch 선택
```

합니다.

즉,

> **학습은 15 Epoch 전체를 수행하지만, 최종 평가에는 Validation Accuracy가 가장 높았던 모델을 사용합니다.**

이 방식은 이전 웹캠 학습 Notebook에서 사용한 실험 설계와 동일합니다. fileciteturn15file10L1-L1


In [11]:
# ============================================================
# 12. 결과 저장 폴더 준비
# ============================================================

if RESULT_DIR.exists():
    shutil.rmtree(
        RESULT_DIR
    )

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ORIGINAL_RESULT_DIR = (
    RESULT_DIR / "original"
)

AUGMENTED_RESULT_DIR = (
    RESULT_DIR / "augmented"
)

ORIGINAL_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

AUGMENTED_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("결과 저장 위치:")
print(RESULT_DIR)


결과 저장 위치:
/content/hair_model_results


# 13. 학습 및 평가 함수

Original과 Augmented 모두 완전히 동일한 학습 함수를 사용합니다.

각 실험에서:

1. Dataset 생성
2. EfficientNet-B0 생성
3. 15 Epoch 학습
4. `val_accuracy`가 가장 높은 모델 저장
5. Best model weight 복원
6. 원본 Test Dataset 평가
7. Classification Report 생성
8. Confusion Matrix 생성
9. Accuracy/Loss curve 생성
10. 모델 및 학습 기록 저장

을 수행합니다.

### 평가 지표

#### Accuracy

전체 Test 이미지 중 올바르게 분류한 비율입니다.

#### Precision

특정 클래스로 예측한 이미지 중 실제로 해당 클래스인 비율입니다.

#### Recall

실제 특정 클래스 이미지 중 모델이 해당 클래스로 찾아낸 비율입니다.

#### F1-score

Precision과 Recall의 조화평균입니다.

#### Macro F1

모든 클래스를 동일한 가중치로 평균낸 F1-score입니다.

두피 클래스별 데이터 수가 완전히 같지 않거나 특정 클래스의 성능이 상대적으로 낮을 수 있기 때문에 전체 Accuracy만 보는 것보다 Macro F1을 함께 확인하는 것이 중요합니다.


In [12]:
# ============================================================
# 13. 학습 + 평가 함수
# ============================================================

def train_and_evaluate(
    model_name,
    train_ds,
    val_ds,
    test_ds,
    result_dir
):

    print()
    print("=" * 75)
    print(f"{model_name} 모델 학습 시작")
    print("=" * 75)

    model = build_model()

    best_model_path = (
        result_dir / "best_model.keras"
    )

    csv_log_path = (
        result_dir / "training_log.csv"
    )

    checkpoint = ModelCheckpoint(
        filepath=str(
            best_model_path
        ),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    )

    csv_logger = CSVLogger(
        str(csv_log_path),
        append=False
    )

    # Early Stopping 없이 정확히 15 Epoch 수행
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=[
            checkpoint,
            csv_logger
        ],
        verbose=1
    )

    # Validation Accuracy가 가장 좋았던 모델을 다시 불러옴
    best_model = keras.models.load_model(
        best_model_path
    )

    best_epoch = (
        int(
            np.argmax(
                history.history["val_accuracy"]
            )
        )
        + 1
    )

    best_val_accuracy = max(
        history.history["val_accuracy"]
    )

    print()
    print(
        f"Best Epoch: {best_epoch}"
    )

    print(
        f"Best Val Accuracy: "
        f"{best_val_accuracy * 100:.2f}%"
    )

    # --------------------------------------------------------
    # Test prediction
    # --------------------------------------------------------

    y_true = []
    y_pred = []

    for images, labels in test_ds:

        probabilities = (
            best_model.predict(
                images,
                verbose=0
            )
        )

        predictions = np.argmax(
            probabilities,
            axis=1
        )

        y_true.extend(
            labels.numpy()
        )

        y_pred.extend(
            predictions
        )

    y_true = np.array(
        y_true
    )

    y_pred = np.array(
        y_pred
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    test_accuracy = accuracy_score(
        y_true,
        y_pred
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro"
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
        target_names=CLASS_NAMES,
        digits=6,
        zero_division=0
    )

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )

    print()
    print("=" * 75)
    print(f"{model_name} Classification Report")
    print("=" * 75)
    print(report_text)

    print("=" * 75)
    print(f"{model_name} 결과")
    print("=" * 75)

    print(
        f"Best Epoch        : "
        f"{best_epoch}"
    )

    print(
        f"Best Val Accuracy : "
        f"{best_val_accuracy * 100:.2f}%"
    )

    print(
        f"Test Accuracy     : "
        f"{test_accuracy * 100:.2f}%"
    )

    print(
        f"Macro F1          : "
        f"{macro_f1:.4f}"
    )

    # --------------------------------------------------------
    # Confusion Matrix
    # --------------------------------------------------------

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        )
    )

    plt.figure(
        figsize=(9, 8)
    )

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES
    )

    plt.xlabel(
        "Predicted"
    )

    plt.ylabel(
        "True"
    )

    plt.title(
        f"{model_name} Confusion Matrix"
    )

    plt.tight_layout()

    cm_path = (
        result_dir
        / "confusion_matrix.png"
    )

    plt.savefig(
        cm_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

    # --------------------------------------------------------
    # Accuracy curve
    # --------------------------------------------------------

    epochs_range = range(
        1,
        len(
            history.history["accuracy"]
        ) + 1
    )

    plt.figure(
        figsize=(8, 5)
    )

    plt.plot(
        epochs_range,
        history.history["accuracy"],
        label="Train Accuracy"
    )

    plt.plot(
        epochs_range,
        history.history["val_accuracy"],
        label="Val Accuracy"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(
        f"{model_name} Accuracy"
    )
    plt.legend()
    plt.grid(True)

    plt.tight_layout()

    accuracy_curve_path = (
        result_dir
        / "accuracy_curve.png"
    )

    plt.savefig(
        accuracy_curve_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

    # --------------------------------------------------------
    # Loss curve
    # --------------------------------------------------------

    plt.figure(
        figsize=(8, 5)
    )

    plt.plot(
        epochs_range,
        history.history["loss"],
        label="Train Loss"
    )

    plt.plot(
        epochs_range,
        history.history["val_loss"],
        label="Val Loss"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(
        f"{model_name} Loss"
    )
    plt.legend()
    plt.grid(True)

    plt.tight_layout()

    loss_curve_path = (
        result_dir
        / "loss_curve.png"
    )

    plt.savefig(
        loss_curve_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

    # --------------------------------------------------------
    # Classification report CSV
    # --------------------------------------------------------

    report_df = pd.DataFrame(
        report_dict
    ).transpose()

    report_df.to_csv(
        result_dir
        / "classification_report.csv",
        encoding="utf-8-sig"
    )

    # --------------------------------------------------------
    # Configuration 저장
    # --------------------------------------------------------

    config = {
        "model": "EfficientNetB0",
        "weights": "ImageNet",
        "image_size": list(
            IMAGE_SIZE
        ),
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "seed": SEED,
        "early_stopping": False,
        "best_epoch": best_epoch,
        "best_val_accuracy": float(
            best_val_accuracy
        ),
        "test_accuracy": float(
            test_accuracy
        ),
        "macro_f1": float(
            macro_f1
        ),
        "class_names": CLASS_NAMES
    }

    with open(
        result_dir / "training_config.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            config,
            f,
            ensure_ascii=False,
            indent=2
        )

    # --------------------------------------------------------
    # History JSON
    # --------------------------------------------------------

    history_json = {
        key: [
            float(v)
            for v in values
        ]
        for key, values
        in history.history.items()
    }

    with open(
        result_dir / "training_history.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            history_json,
            f,
            ensure_ascii=False,
            indent=2
        )

    # --------------------------------------------------------
    # 클래스 매핑 저장
    # --------------------------------------------------------

    class_map = {
        str(i): name
        for i, name
        in enumerate(CLASS_NAMES)
    }

    with open(
        result_dir / "class_names.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            class_map,
            f,
            ensure_ascii=False,
            indent=2
        )

    return {
        "model_name": model_name,
        "model": best_model,
        "best_epoch": best_epoch,
        "best_val_accuracy": best_val_accuracy,
        "test_accuracy": test_accuracy,
        "macro_f1": macro_f1,
        "report": report_dict,
        "confusion_matrix": cm,
        "history": history.history
    }


# 14. Original Dataset 학습

먼저 증강을 사용하지 않은 Original Dataset으로 baseline 모델을 학습합니다.

이 결과가 이후 Augmented 모델과 비교할 **기준선(baseline)** 이 됩니다.

```text
Original Train
      ↓
EfficientNet-B0
      ↓
Original Model
      ↓
Original Test
```

이 결과를 통해 증강을 적용하지 않았을 때 두피 현미경 데이터에서 어느 정도 성능을 얻을 수 있는지 확인합니다.


In [ ]:
# ============================================================
# 14. Original Dataset 학습
# ============================================================

original_train_ds, original_val_ds, original_test_ds = (
    make_datasets(
        ORIGINAL_DIR
    )
)

original_result = train_and_evaluate(
    model_name="Original",
    train_ds=original_train_ds,
    val_ds=original_val_ds,
    test_ds=original_test_ds,
    result_dir=ORIGINAL_RESULT_DIR
)


Found 11600 files belonging to 5 classes.
Found 1450 files belonging to 5 classes.
Found 1450 files belonging to 5 classes.

Original 모델 학습 시작
Epoch 1/15


# 15. Augmented Dataset 학습

다음으로 Augmented Dataset을 학습합니다.

중요한 점은:

```text
Augmented Train
= 원본 + 증강

Augmented Val
= 원본

Augmented Test
= 원본
```

이라는 것입니다.

따라서 모델은 Train에서 더 다양한 촬영 조건을 경험하지만, 최종 평가는 원본 이미지로 수행됩니다.

이 결과를 Original 모델과 비교하면 **데이터 증강이 두피 현미경 영상의 일반화 성능에 미친 효과**를 확인할 수 있습니다.


In [ ]:
# ============================================================
# 15. Augmented Dataset 학습
# ============================================================

augmented_train_ds, augmented_val_ds, augmented_test_ds = (
    make_datasets(
        AUGMENTED_DIR
    )
)

augmented_result = train_and_evaluate(
    model_name="Augmented",
    train_ds=augmented_train_ds,
    val_ds=augmented_val_ds,
    test_ds=augmented_test_ds,
    result_dir=AUGMENTED_RESULT_DIR
)


# 16. Original vs Augmented 최종 비교

두 모델의 결과를 한 표로 비교합니다.

특히 다음 세 가지를 중요하게 확인합니다.

### Best Validation Accuracy

학습 중 Validation 데이터에서 가장 높은 정확도입니다.

### Test Accuracy

학습에 사용하지 않은 Test 데이터에 대한 최종 분류 정확도입니다.

### Macro F1

각 클래스를 동일한 중요도로 취급하여 계산한 평균 F1-score입니다.

두피 질환별 성능이 균등한지를 확인할 때 중요합니다.

---

## 결과 해석

예를 들어:

```text
Original
Test Accuracy = 80%
Macro F1      = 0.78

Augmented
Test Accuracy = 84%
Macro F1      = 0.83
```

이라면 증강을 적용한 모델이 전체적인 일반화 성능뿐 아니라 클래스 간 균형도 개선되었다고 해석할 수 있습니다.

반대로 Accuracy만 상승하고 Macro F1이 하락한다면 특정 클래스에 대한 성능만 좋아졌을 가능성이 있으므로 Confusion Matrix를 함께 확인해야 합니다.


In [ ]:
# ============================================================
# 16. Original vs Augmented 비교
# ============================================================

comparison = pd.DataFrame({
    "Model": [
        "Original",
        "Augmented"
    ],
    "Best Val Accuracy": [
        original_result["best_val_accuracy"],
        augmented_result["best_val_accuracy"]
    ],
    "Test Accuracy": [
        original_result["test_accuracy"],
        augmented_result["test_accuracy"]
    ],
    "Macro F1": [
        original_result["macro_f1"],
        augmented_result["macro_f1"]
    ],
    "Best Epoch": [
        original_result["best_epoch"],
        augmented_result["best_epoch"]
    ]
})

comparison_display = comparison.copy()

comparison_display[
    "Best Val Accuracy"
] *= 100

comparison_display[
    "Test Accuracy"
] *= 100

comparison_display[
    "Best Val Accuracy"
] = comparison_display[
    "Best Val Accuracy"
].map(lambda x: f"{x:.2f}%")

comparison_display[
    "Test Accuracy"
] = comparison_display[
    "Test Accuracy"
].map(lambda x: f"{x:.2f}%")

comparison_display[
    "Macro F1"
] = comparison_display[
    "Macro F1"
].map(lambda x: f"{x:.4f}")

display(
    comparison_display
)

print()
print("=" * 75)
print("Augmented - Original")
print("=" * 75)

print(
    "Test Accuracy 변화: "
    f"{(augmented_result['test_accuracy'] - original_result['test_accuracy']) * 100:+.2f}%p"
)

print(
    "Macro F1 변화: "
    f"{augmented_result['macro_f1'] - original_result['macro_f1']:+.4f}"
)


# 17. 클래스별 F1 비교

전체 Accuracy만으로는 어떤 두피 클래스에서 문제가 발생하는지 알 수 없습니다.

따라서 Original과 Augmented의 클래스별 F1-score를 비교합니다.

예를 들어 특정 클래스의 F1이:

```text
Original   0.65
Augmented  0.78
```

로 상승했다면 해당 클래스에서 증강이 도움이 되었을 가능성이 있습니다.

반대로 특정 클래스의 F1이 감소했다면 증강 과정에서 해당 클래스의 중요한 시각적 특징이 약화되었는지 확인해야 합니다.

특히 두피 현미경 영상에서는 조명이나 색상 변화가 질환의 중요한 시각적 단서를 훼손할 수도 있으므로 **모든 증강이 무조건 좋은 것은 아닙니다.**


In [ ]:
# ============================================================
# 17. 클래스별 F1 비교
# ============================================================

class_rows = []

for class_name in CLASS_NAMES:

    original_f1 = (
        original_result["report"]
        [class_name]["f1-score"]
    )

    augmented_f1 = (
        augmented_result["report"]
        [class_name]["f1-score"]
    )

    class_rows.append({
        "Class": class_name,
        "Original F1": original_f1,
        "Augmented F1": augmented_f1,
        "Change": (
            augmented_f1
            - original_f1
        )
    })

class_comparison = pd.DataFrame(
    class_rows
)

display(
    class_comparison.style.format({
        "Original F1": "{:.4f}",
        "Augmented F1": "{:.4f}",
        "Change": "{:+.4f}"
    })
)


# 18. Confusion Matrix 비교

Confusion Matrix는 어떤 클래스가 어떤 클래스로 잘못 분류되는지 보여줍니다.

행(row)은 실제 클래스이고,

열(column)은 모델이 예측한 클래스입니다.

예:

```text
             예측
           A   B   C
실제 A    80  10   0
실제 B     5  70   5
실제 C     0   8  72
```

대각선 값이 클수록 정확하게 분류된 것이고, 대각선 밖의 값은 오분류입니다.

두피 데이터에서는 비슷한 시각적 특징을 가진 클래스 사이에서 오분류가 집중될 수 있으므로 Confusion Matrix를 통해 확인합니다.


In [ ]:
# ============================================================
# 18. 두 모델 Confusion Matrix 비교
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 7)
)

sns.heatmap(
    original_result["confusion_matrix"],
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    ax=axes[0]
)

axes[0].set_title(
    "Original"
)

axes[0].set_xlabel(
    "Predicted"
)

axes[0].set_ylabel(
    "True"
)

sns.heatmap(
    augmented_result["confusion_matrix"],
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    ax=axes[1]
)

axes[1].set_title(
    "Augmented"
)

axes[1].set_xlabel(
    "Predicted"
)

axes[1].set_ylabel(
    "True"
)

plt.tight_layout()
plt.show()


# 19. 최종 모델 선택

최종 모델은 단순히 Validation Accuracy가 높은 모델을 선택하는 것이 아니라 다음을 종합적으로 확인합니다.

1. Test Accuracy
2. Macro F1
3. 클래스별 Recall
4. 클래스별 F1
5. Confusion Matrix
6. 학습 곡선
7. 실제 현미경 촬영 이미지에 대한 추론 결과

특히 의료 스크리닝 모델에서는 특정 클래스의 Recall이 지나치게 낮으면 전체 Accuracy가 높더라도 문제가 될 수 있습니다.

따라서:

```text
Accuracy가 높다
        ≠
모든 질환을 균등하게 잘 분류한다
```

는 점을 고려합니다.

최종적으로 Original과 Augmented 중 더 적절한 모델을 선택하여 이후 실제 USB 현미경 추론 단계에 사용합니다.


In [ ]:
# ============================================================
# 19. 자동 추천 결과
# ============================================================

if (
    augmented_result["macro_f1"]
    >
    original_result["macro_f1"]
):
    recommended = "Augmented"
else:
    recommended = "Original"

print("=" * 75)
print("현재 지표 기준 추천 모델")
print("=" * 75)
print(recommended)

print()
print("단, 최종 결정 전 클래스별 Recall/F1과")
print("Confusion Matrix 및 실제 현미경 촬영 테스트를 함께 확인하세요.")


# 20. 최종 모델 파일 확인

각 모델은 다음과 같은 구조로 저장됩니다.

```text
hair_model_results/
│
├── original/
│   ├── best_model.keras
│   ├── training_log.csv
│   ├── training_history.json
│   ├── training_config.json
│   ├── class_names.json
│   ├── classification_report.csv
│   ├── confusion_matrix.png
│   ├── accuracy_curve.png
│   └── loss_curve.png
│
└── augmented/
    ├── best_model.keras
    ├── training_log.csv
    ├── training_history.json
    ├── training_config.json
    ├── class_names.json
    ├── classification_report.csv
    ├── confusion_matrix.png
    ├── accuracy_curve.png
    └── loss_curve.png
```

이렇게 저장하면 나중에 단순히 모델 파일만 남는 것이 아니라 **학습 조건과 성능 결과를 함께 보관**할 수 있습니다.


In [ ]:
# ============================================================
# 20. 결과 파일 확인
# ============================================================

print("===== Original 결과 =====")

for p in sorted(
    ORIGINAL_RESULT_DIR.iterdir()
):

    if p.is_file():

        size_mb = (
            p.stat().st_size
            / (1024 * 1024)
        )

        print(
            f"{p.name:35s}"
            f"{size_mb:8.2f} MB"
        )


print()
print("===== Augmented 결과 =====")

for p in sorted(
    AUGMENTED_RESULT_DIR.iterdir()
):

    if p.is_file():

        size_mb = (
            p.stat().st_size
            / (1024 * 1024)
        )

        print(
            f"{p.name:35s}"
            f"{size_mb:8.2f} MB"
        )


# 21. Google Drive에 결과 백업

Colab의 `/content` 영역은 런타임이 종료되면 삭제될 수 있습니다.

따라서 학습이 완료된 후에는 모델과 결과 파일을 Google Drive에 백업합니다.

저장 위치:

```text
My Drive/
└── hair_dataset/
    ├── hair_processed.zip
    │
    └── hair_model_results/
        ├── original/
        └── augmented/
```

원본 ZIP과 학습 결과를 별도 폴더로 관리하여 데이터셋과 모델 결과가 섞이지 않도록 합니다.


In [ ]:
# ============================================================
# 21. Google Drive 결과 백업
# ============================================================

if DRIVE_RESULT_DIR.exists():
    shutil.rmtree(
        DRIVE_RESULT_DIR
    )

shutil.copytree(
    RESULT_DIR,
    DRIVE_RESULT_DIR
)

print("Google Drive 백업 완료")
print()
print(DRIVE_RESULT_DIR)

print()
print("파일 목록:")

for p in sorted(
    DRIVE_RESULT_DIR.rglob("*")
):

    if p.is_file():

        relative = p.relative_to(
            DRIVE_RESULT_DIR
        )

        print(
            " -",
            relative
        )


# 22. 최종 성능 요약 파일 저장

보고서나 발표 자료에서 바로 사용할 수 있도록 Original과 Augmented의 핵심 성능을 CSV로 저장합니다.

포함 항목:

- Best Epoch
- Best Validation Accuracy
- Test Accuracy
- Macro F1

추가로 클래스별 Precision / Recall / F1은 각 모델의 `classification_report.csv`에서 확인할 수 있습니다.


In [ ]:
# ============================================================
# 22. 최종 요약 CSV 저장
# ============================================================

summary_df = pd.DataFrame([
    {
        "Model": "Original",
        "Best Epoch": original_result["best_epoch"],
        "Best Val Accuracy": original_result["best_val_accuracy"],
        "Test Accuracy": original_result["test_accuracy"],
        "Macro F1": original_result["macro_f1"]
    },
    {
        "Model": "Augmented",
        "Best Epoch": augmented_result["best_epoch"],
        "Best Val Accuracy": augmented_result["best_val_accuracy"],
        "Test Accuracy": augmented_result["test_accuracy"],
        "Macro F1": augmented_result["macro_f1"]
    }
])

summary_path = (
    RESULT_DIR / "final_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

shutil.copy2(
    summary_path,
    DRIVE_RESULT_DIR / summary_path.name
)

display(
    summary_df.style.format({
        "Best Val Accuracy": "{:.4f}",
        "Test Accuracy": "{:.4f}",
        "Macro F1": "{:.4f}"
    })
)

print()
print("저장:", DRIVE_RESULT_DIR / summary_path.name)


# 23. 최종 정리

이번 Notebook에서 수행한 전체 과정은 다음과 같습니다.

```text
Google Drive
    │
    │ hair_processed.zip
    ↓
Colab /content로 복사
    ↓
압축 해제
    ↓
original / augmented 구조 확인
    ↓
클래스 및 이미지 개수 검증
    ↓
TensorFlow Dataset 생성
    ↓
EfficientNet-B0
    │
    ├── Original Train
    │       ↓
    │   Original Model
    │
    └── Augmented Train
            ↓
        Augmented Model
                │
                ↓
       동일한 원본 Test 평가
                │
                ↓
      Original vs Augmented 비교
                │
                ↓
      최종 두피 현미경 모델 선택
```

---

## 이번 실험에서 기억해야 할 핵심

### 데이터 증강

```text
Original Train
→ 원본

Augmented Train
→ 원본 + 증강

Val
→ 원본

Test
→ 원본
```

### 학습

```text
EfficientNet-B0
ImageNet pretrained
224×224
Batch 32
15 Epoch
Early Stopping 없음
```

### 최종 선택

단순히 Accuracy만 비교하지 않고:

```text
Test Accuracy
+
Macro F1
+
Class Recall/F1
+
Confusion Matrix
+
실제 현미경 추론 결과
```

를 함께 확인합니다.

---

## 프로젝트에서의 위치

이 모델은 최종 시스템의 **USB 현미경 기반 두피 전문 분류 모델**에 해당합니다.

```text
USB 현미경
     ↓
두피 확대 이미지
     ↓
EfficientNet-B0
     ↓
두피 상태/질환 6-class 분류
     ↓
전문 분류 결과
     ↓
향후 의료 MLLM/VLM의 입력으로 활용
```

프로젝트 계획에서도 안구·피부·두피 질환별 전문 분류 모델을 구축하고, 이후 질환군별 전문 모델을 호출하는 라우팅 구조로 연결하는 방향을 목표로 하고 있습니다. fileciteturn17file15L1-L1

또한 의료 AI의 최종 결과는 확정 진단이 아니라 **스크리닝 보조 및 검토 권고**를 위한 결과로 사용하는 방향입니다. fileciteturn15file3L1-L1


# 24. 학습 완료 후 다음 단계

학습이 끝나면 다음 순서로 진행합니다.

### 1. Original / Augmented 성능 비교

먼저 두 모델 중 어느 쪽이 더 좋은지 확인합니다.

### 2. Confusion Matrix 분석

어떤 두피 클래스끼리 혼동되는지 확인합니다.

### 3. 실제 USB 현미경 촬영

공개 데이터셋과 다른 실제 카메라 환경에서 이미지를 촬영합니다.

예:

```text
USB 현미경
    ↓
실제 두피 촬영
    ↓
224×224 Resize
    ↓
선정된 EfficientNet-B0
    ↓
6-class prediction
```

### 4. Domain Gap 확인

공개 데이터셋 Test Accuracy와 실제 USB 현미경 이미지의 성능 차이를 확인합니다.

이 차이가 크다면 다음 단계에서 현미경 촬영 데이터를 추가로 확보하여 fine-tuning하거나, 현미경 환경에 맞는 추가 증강을 조정할 수 있습니다.

### 5. 최종 통합

최종적으로 프로젝트에서는:

```text
안구 전문 모델
피부 현미경 전문 모델
피부 웹캠 전문 모델
두피 현미경 전문 모델
        ↓
전문 모델 Router
        ↓
의료 MLLM / VLM
        ↓
스크리닝 보조 결과
```

로 연결하는 방향으로 발전시킬 수 있습니다.


# 25. 완료

이 Notebook을 처음부터 끝까지 실행하면:

1. Google Drive 연결
2. `hair_processed.zip` 확인
3. Colab으로 ZIP 복사
4. 압축 해제
5. `original / augmented` 자동 탐색
6. 클래스 및 이미지 개수 확인
7. 데이터 구조 검증
8. TensorFlow Dataset 생성
9. EfficientNet-B0 생성
10. Original 15 Epoch 학습
11. Augmented 15 Epoch 학습
12. Validation Accuracy 기준 Best Model 저장
13. 원본 Test 평가
14. Classification Report 생성
15. Confusion Matrix 생성
16. Accuracy/Loss 그래프 생성
17. Original vs Augmented 비교
18. 클래스별 F1 비교
19. 최종 결과 CSV 저장
20. Google Drive에 모델 및 결과 백업

까지 수행합니다.

**이 Notebook의 최종 목적은 단순히 정확도가 높은 모델 하나를 만드는 것이 아니라, `현미경 두피 데이터에서 증강이 실제 일반화 성능을 개선하는지 실험적으로 비교하고, 그 결과를 재현 가능한 형태로 저장하는 것`입니다.**
